<div style="text-align: center; padding-top: 30px; padding-bottom: 10px;">

<h1 style="font-size: 2.8em; font-weight: 600; margin-bottom: 0.2em;">
Global Neural Network Model
</h1>

<p style="font-size: 1.2em; color: gray; font-style: italic; margin-top: 0;">
This notebook visualises the main results from the paper.
</p>

</div>

In [20]:
import sys, os, yaml
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path


for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src_dir = parent / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        sys.path.insert(0, str(parent / "scripts"))   # hdf5_min lives in scripts/
        break
    
    
from models.helper_functions.global_model.prepare_data import Prepare
from utils import create_pred_input, add_histogram
from hdf5_min import read_weights

################################################## User Input ##################################################
n_models = 10
run_dir  = '../runs/estimation/2026-06-13_22-11-06_global_IC_country_trends'
###############################################################################################################

# Hydra config saved with the estimation run, flattened the way the model expects.
with open(f'{run_dir}/.hydra/config.yaml', "r") as file:
    cfg = yaml.safe_load(file)['instance']

# cfg = {
#     **run_cfg['optim'],
#     **run_cfg['model'],
#     **run_cfg['server'],
# }
assert 'input_vars' in cfg, 'The flattened run config must define input_vars'

print(Path(run_dir).name)
for k in ("target_mode", "within_projection", "dynamic_model", "country_trends",
          "quadratic_trends", "data_source", "data_end"):
    print(f"  {k:20} {cfg.get(k)}")
print(f"  {'architectures':20} {len(cfg.get('nodes_list', []))}")

assert str(cfg.get("target_mode", "growth")).lower() == "growth", \
    "this notebook expects a growth run"
assert bool(cfg.get("within_projection", False)), \
    "the numpy forward pass below assumes within_projection, i.e. the saved graph is the climate net alone"


2026-06-13_22-11-06_global_IC_country_trends
  target_mode          growth
  within_projection    True
  dynamic_model        False
  country_trends       True
  quadratic_trends     True
  data_source          wb
  data_end             2024
  architectures        55


## 1. Data preperation

In [2]:

#load data and create prediction grid
data=pd.read_excel('../data/MainData.xlsx')
growth, precip, temp = Prepare(data, data_source=cfg["data_source"])
x_train = {0:temp, 1:precip}

mean_temp=np.nanmean(data["TempPopWeight"])
std_temp=np.nanstd(data["TempPopWeight"])
mean_precip=np.nanmean(data["PrecipPopWeight"])
std_precip=np.nanstd(data["PrecipPopWeight"])


#prediction grid for plotting 
pred_input, T, P= create_pred_input(mc=False, mean_T=mean_temp, std_T=std_temp, mean_P=mean_precip, std_P=std_precip, precip_capped=True)


## 2. Model overview

In [21]:
results = dict(np.load(f'{run_dir}/results.npy', allow_pickle=True).item())
results = {k: v for k, v in results.items() if v is not None}

# results.npy stores [BIC, AIC] per architecture; BIC is index 0 and is what
# selects the model in the paper. The old key was index 2, a third element that
# this format no longer carries.
top_models = sorted(results, key=lambda node: results[node][0])[:n_models]

print(f"{len(results)} architectures, showing the {len(top_models)} best by BIC\n")
for rank, node in enumerate(top_models, 1):
    bic, aic = float(results[node][0]), float(results[node][1])
    print(f"  {rank:>2}. {str(node):<14} BIC {bic:>12.1f}   AIC {aic:>12.1f}")


55 architectures, showing the 10 best by BIC

   1. (2,)           BIC     -53270.4   AIC     -58021.3
   2. (2, 2)         BIC     -53238.6   AIC     -58032.9
   3. (4,)           BIC     -53215.8   AIC     -58024.7
   4. (2, 2, 2)      BIC     -53199.6   AIC     -58037.4
   5. (4, 2)         BIC     -53172.4   AIC     -58039.2
   6. (4, 2, 2)      BIC     -53137.1   AIC     -58047.4
   7. (4, 4)         BIC     -53102.6   AIC     -58056.3
   8. (8,)           BIC     -53094.9   AIC     -58019.6
   9. (4, 4, 2)      BIC     -53047.1   AIC     -58058.7
  10. (8, 2, 2)      BIC     -53039.1   AIC     -58094.2


## 3. Prediction on Temperature-Precipitation grid

In [30]:
# Surfaces are evaluated with a plain numpy forward pass over the saved weights
# rather than by rebuilding a Keras model and calling load_weights.
#
# Why: this run has within_projection = true, so the saved graph is the climate
# net alone (hidden Dense layers with a bias, then a use_bias=False output); the
# fixed effects and country trends are recovered by OLS and never enter the
# graph. Keras 3 keys weights by autogenerated layer name, so rebuilding the
# model from a config that differs by even one layer misaligns every tensor and
# fails with "Layer 'dense_1' expected 1 variables, but received 2". Reading the
# weights directly removes that whole failure mode, and is the same code path
# the levels notebook uses, so the two notebooks are comparable by construction.

############################## User Input ##############################
CENTRE_AT_SAMPLE_MEAN = True   # shift each surface to zero at mean (T, P)
########################################################################


def swish(z):
    return z / (1.0 + np.exp(-np.clip(z, -60.0, 60.0)))


def forward(w, X):
    """Chained Dense stack: hidden layers carry a bias, the output layer does not."""
    idx = sorted({int(k.split("/")[1].split("_")[1]) if "_" in k.split("/")[1] else 0
                  for k in w if k.startswith("layers")})
    h = X
    for i in idx:
        b = f"layers/dense{'' if i == 0 else f'_{i}'}/vars"
        h = h @ np.asarray(w[f"{b}/0"], float)
        if f"{b}/1" in w:
            h = swish(h + np.asarray(w[f"{b}/1"], float))
    return h[..., 0]


# The network takes standardised inputs, so the all-zeros input IS the sample
# mean climate. Subtracting f(0, 0) makes each surface pass through zero there,
# and the figure then shows growth differences relative to the average climate
# rather than absolute levels. The shift is a single scalar per architecture, so
# the shape of the surface is untouched; it is also the natural normalisation
# here, since an additive constant in the climate net is not separately
# identified from the country fixed effects.
ORIGIN = np.zeros((1,) * (pred_input.ndim - 1) + (pred_input.shape[-1],), dtype=float)


def growth_surface(node):
    w = read_weights(f'{run_dir}/parameters/{node}.weights.h5')
    Z = forward(w, pred_input).reshape(T.shape)
    if CENTRE_AT_SAMPLE_MEAN:
        Z = Z - float(np.ravel(forward(w, ORIGIN))[0])+0.05
    return Z


surfaces = {}
for node in top_models:
    path = f'{run_dir}/parameters/{node}.weights.h5'
    if not os.path.exists(path):
        print(f"  {str(node):>12}  no weights file, skipped")
        continue
    surfaces[node] = growth_surface(node)

if not surfaces:
    raise RuntimeError(f"no weight files found under {run_dir}/parameters")

z = np.mean(list(surfaces.values()), axis=0).reshape(T.shape)

if CENTRE_AT_SAMPLE_MEAN:
    print(f"centred at the sample mean climate: "
          f"T = {mean_temp:.2f} C, P = {mean_precip/1000:.3f} m")
    # the mean surface is an average of centred surfaces, so it is centred too
    i0 = int(np.argmin(np.abs(T[0, :] - mean_temp)))
    j0 = int(np.argmin(np.abs(P[:, 0] - mean_precip)))
    print(f"  value at nearest grid node ({T[j0, i0]:.2f} C, "
          f"{P[j0, i0]/1000:.3f} m): {z[j0, i0]:+.5f}\n")

for node, Z in surfaces.items():
    print(f"  {str(node):>12}  range {Z.min():+.4f} to {Z.max():+.4f}")
print(f"  {'mean':>12}  range {z.min():+.4f} to {z.max():+.4f}  (n={len(surfaces)})")

model_surfaces = [
    go.Surface(x=T, y=P / 1000, z=Z, colorscale='Cividis',
               opacity=0.85, showscale=False, name=f'Model {node}')
    for node, Z in surfaces.items()
]

mean_surface = go.Surface(
    x=T, y=P / 1000, z=z,          # P is in mm on the grid, plotted in m
    colorscale='Cividis',
    opacity=0.85,
    showscale=False,
    name='mean_surface',
)


# --- previous Keras path, kept for reference -------------------------------
# Fails on within_projection runs, see the note at the top of this cell.
#
# from models import MultivariateModelGlobal as Model
# model_surfaces = []
# for idx, node in enumerate(top_models[0:n_models], 1):
#     factory = Model(node, cfg, x_train, growth)
#     factory.Depth = len(node)
#     model = factory.get_model()
#     model.load_params(f'{run_dir}/parameters/{node}.weights.h5')
#     pred_flat = model.model_visual.predict([pred_input]).reshape(-1,)
#     Growth = pred_flat.reshape(T.shape)
#     model_surfaces.append(go.Surface(x=T, y=P/1000, z=Growth,
#                                      colorscale='Cividis', opacity=0.85,
#                                      showscale=False, name=f'Model {node}'))
# z = np.mean([s.z for s in model_surfaces], axis=0).reshape(T.shape)


centred at the sample mean climate: T = 20.31 C, P = 1.279 m
  value at nearest grid node (20.22 C, 1.288 m): +0.05038

          (2,)  range -0.0698 to +0.0504
        (2, 2)  range -0.0604 to +0.0817
          (4,)  range -0.1088 to +0.0526
     (2, 2, 2)  range -0.1219 to +0.0650
        (4, 2)  range -0.1269 to +0.0986
     (4, 2, 2)  range -0.0857 to +0.0891
        (4, 4)  range -0.1046 to +0.1038
          (8,)  range -0.0965 to +0.0789
     (4, 4, 2)  range -0.0965 to +0.1012
     (8, 2, 2)  range -0.0365 to +0.1307
          mean  range -0.0719 to +0.0667  (n=10)


## 4. Data visualization

In [31]:

plot_data=model_surfaces[0]
fig= go.Figure(data=plot_data)

## 3-d surface plot
fig.update_layout(
        autosize=True,
       
        margin=dict(
            l=0,
            r=0,
            b=0,
            t=0,
        ),

            scene=dict(
                xaxis_title='Temperature (°C)',
                yaxis_title='Precipitation (m)',
                zaxis=dict(title=dict(text="Δ ln(GDP)"),range=[-0.3, 0.3]),
                camera=dict(eye=dict(x=1.738, y=-1.780, z=0.589))
                
            ),
            
            legend=dict(
                bgcolor='rgba(255,255,255,0.7)',
                bordercolor='black',
                borderwidth=1
            ),
            font=dict(
            size=10
        )
        )
fig.show()


In [32]:
fig=add_histogram(fig, data, legend=False, precip_capped=True)
#Let figure pass 0 in the 
fig.show()

In [41]:
# Export. The PDF is cropped on the way out, so no Acrobat "Set Page Box" step
# is needed.
#
# The crop is taken from a reference figure rather than from margin numbers:
# Plotly writes every one of these surfaces on the same 1350 x 1050 pt page with
# the same camera, so copying the reference's CropBox reproduces its framing
# exactly and keeps the whole Global/Regional figure set visually consistent. If
# the reference is missing, it falls back to the pdf-crop defaults.
from pypdf import PdfReader, PdfWriter

CROP_REFERENCE = ROOT / "paper/Figures/Global/global_top10surfaces_CT_fit.pdf"

# Nudges applied to the reference box, in points. Negative moves an edge
# outwards (less crop), positive inwards (more crop). The reference was framed
# for the ten-surface overlay; this figure carries a z-axis title, so the left
# edge needs extra room, and the right needs room for the precipitation label.
# Edit THESE to adjust the framing. MARGINS_MM below is only the fallback used
# when the reference file is missing, and has no effect while it exists.
CROP_ADJUST_PT = {"left": -40.0, "bottom": 0.0, "right": 30.0, "top": 30.0}
MARGINS_MM = {"top": 70.0, "bottom": 28.6, "left": 75.0, "right": 112}
MM_TO_PT = 72.0 / 25.4


def reference_box(path, adjust=None):
    """CropBox of the reference, nudged, in the original page coordinate system."""
    a = adjust or {}
    box = PdfReader(str(path)).pages[0].cropbox
    return (float(box.left) + a.get("left", 0.0),
            float(box.bottom) + a.get("bottom", 0.0),
            float(box.right) + a.get("right", 0.0),
            float(box.top) - a.get("top", 0.0))


def crop_pdf(path, reference=CROP_REFERENCE, margins_mm=MARGINS_MM):
    """Set CropBox and MediaBox in place, matching `reference` when available."""
    box = reference_box(reference, CROP_ADJUST_PT) if reference and Path(reference).exists() else None

    reader, writer = PdfReader(str(path)), PdfWriter()
    for page in reader.pages:
        page_box = page.mediabox
        if box is not None:
            left, bottom, right, top = box
        else:
            left = float(page_box.left) + margins_mm["left"] * MM_TO_PT
            bottom = float(page_box.bottom) + margins_mm["bottom"] * MM_TO_PT
            right = float(page_box.right) - margins_mm["right"] * MM_TO_PT
            top = float(page_box.top) - margins_mm["top"] * MM_TO_PT
        if right <= left or top <= bottom:
            raise ValueError(f"crop box is empty for {path}")
        page.cropbox.lower_left = (left, bottom)
        page.cropbox.upper_right = (right, top)
        page.mediabox.lower_left = (left, bottom)
        page.mediabox.upper_right = (right, top)
        writer.add_page(page)
    with open(path, "wb") as fh:
        writer.write(fh)
    return (left, bottom, right, top), box is not None


html_dir = ROOT / "results/images"
html_dir.mkdir(parents=True, exist_ok=True)
html = html_dir / f"global_mean_CT_{Path(run_dir).name}.html"
fig.write_html(html, include_plotlyjs="cdn")

pdf_dir = ROOT / "paper/Figures/Global"
pdf_dir.mkdir(parents=True, exist_ok=True)
pdf = pdf_dir / "global_mean_CT_fit.pdf"      # included at 5_Results.tex:39

# width/height/scale here produce a 1350 x 1050 pt page, which is the page the
# reference CropBox was measured on. Changing them silently reframes the figure,
# so the page size is checked rather than assumed.
fig.write_image(pdf, width=900, height=700, scale=2)

_pg = PdfReader(str(pdf)).pages[0].mediabox
_w, _h = float(_pg.width), float(_pg.height)
if abs(_w - 1350) > 1 or abs(_h - 1050) > 1:
    print(f"   WARNING: page is {_w:.0f} x {_h:.0f} pt, not 1350 x 1050; "
          f"the reference crop box will not frame this figure correctly")

(l, b, r, t), matched = crop_pdf(pdf)

print(f"-> {html}")
print(f"-> {pdf}")
print(f"   crop [{l:.3f} {b:.4f} {r:.3f} {t:.3f}]  "
      f"({r - l:.1f} x {t - b:.1f} pt)  "
      + (f"matched {CROP_REFERENCE.name}" if matched else "margin fallback"))


Resorting to unclean kill browser.


-> c:\Cowork\dev\Clim_Growth\Paper_1\results\images\global_mean_CT_2026-06-13_22-11-06_global_IC_country_trends.html
-> c:\Cowork\dev\Clim_Growth\Paper_1\paper\Figures\Global\global_mean_CT_fit.pdf
   crop [215.123 81.0285 1061.103 821.571]  (846.0 x 740.5 pt)  matched global_top10surfaces_CT_fit.pdf
